In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

df_missing = pd.read_csv('data/processed/phishing_missing.csv')

X_missing = df_missing.drop('Result', axis=1)
y = df_missing['Result']

print(f"Liczba braków w X_missing przed imputacją:\n{X_missing.isnull().sum().head(5)}")

Liczba braków w X_missing przed imputacją:
having_IP_Address           1105
URL_Length                  1105
Shortining_Service          1105
having_At_Symbol            1105
double_slash_redirecting    1105
dtype: int64


In [2]:
from sklearn.impute import SimpleImputer, KNNImputer


# Sposób 1: Most Frequent
imputer_mode = SimpleImputer(strategy='most_frequent')
X_imputed_mode = pd.DataFrame(imputer_mode.fit_transform(X_missing), columns=X_missing.columns)

# Sposób 2: Constant
imputer_constant = SimpleImputer(strategy='constant', fill_value=0)
X_imputed_const = pd.DataFrame(imputer_constant.fit_transform(X_missing), columns=X_missing.columns)

# Sposób 3: KNN
imputer_knn = KNNImputer(n_neighbors=5)
X_imputed_knn_raw = imputer_knn.fit_transform(X_missing)
X_imputed_knn = pd.DataFrame(np.round(X_imputed_knn_raw), columns=X_missing.columns)

df_mode = pd.concat([X_imputed_mode, y], axis=1)
df_const = pd.concat([X_imputed_const, y], axis=1)
df_knn = pd.concat([X_imputed_knn, y], axis=1)

print("Sprawdzenie braków po imputacji KNN:")
print(df_knn.isnull().sum().head(5))

Sprawdzenie braków po imputacji KNN:
having_IP_Address           0
URL_Length                  0
Shortining_Service          0
having_At_Symbol            0
double_slash_redirecting    0
dtype: int64


In [3]:
outliers_mask = ~df_knn.isin([-1, 0, 1, -1.0, 0.0, 1.0])

outliers_count = outliers_mask.sum().sum()
print(f"Liczba wykrytych klasycznych wartości odstających (poza zbiorem -1, 0, 1): {outliers_count}")

for col in df_knn.columns[:5]:
    print(f"Unikalne wartości w kolumnie {col}: {df_knn[col].unique()}")

Liczba wykrytych klasycznych wartości odstających (poza zbiorem -1, 0, 1): 0
Unikalne wartości w kolumnie having_IP_Address: [-1.  1.  0.]
Unikalne wartości w kolumnie URL_Length: [ 1.  0. -1.]
Unikalne wartości w kolumnie Shortining_Service: [ 1. -1.  0.]
Unikalne wartości w kolumnie having_At_Symbol: [ 1. -1.  0.]
Unikalne wartości w kolumnie double_slash_redirecting: [-1.  1.  0.]


In [6]:
from scipy.io import arff

data_orig = arff.loadarff('data/raw/Training Dataset.arff')
df_original = pd.DataFrame(data_orig[0])
for col in df_original.columns:
    if df_original[col].dtype == 'object':
        df_original[col] = df_original[col].str.decode('utf-8')
df_original = df_original.apply(pd.to_numeric)

base_datasets = {
    'original': df_original,
    'mode': df_mode,
    'const': df_const,
    'knn': df_knn
}

def min_max_scaler_manual(df, target_col='Result'):
    df_scaled = df.copy()
    features = [c for c in df.columns if c != target_col]

    for col in features:
        min_val = df_scaled[col].min()
        max_val = df_scaled[col].max()
        if max_val != min_val:
            df_scaled[col] = (df_scaled[col] - min_val) / (max_val - min_val)

    return df_scaled

def standardization_manual(df, target_col='Result'):
    df_scaled = df.copy()
    features = [c for c in df.columns if c != target_col]

    for col in features:
        mean_val = df_scaled[col].mean()
        std_val = df_scaled[col].std()
        if std_val != 0:
            df_scaled[col] = (df_scaled[col] - mean_val) / std_val

    return df_scaled

final_datasets = {}

for name, dataset in base_datasets.items():
    final_datasets[f"{name}_minmax"] = min_max_scaler_manual(dataset)
    final_datasets[f"{name}_std"] = standardization_manual(dataset)

print(f"Pomyślnie wygenerowano {len(final_datasets)} przeskalowanych zbiorów danych!")
print("\nLista utworzonych zbiorów:")
for name in final_datasets.keys():
    print(f"- {name}")

print("\nPrzykład: 5 pierwszych wierszy dla 'original_std' (Standaryzacja):")
display(final_datasets['original_std'].head())

Pomyślnie wygenerowano 8 przeskalowanych zbiorów danych!

Lista utworzonych zbiorów:
- original_minmax
- original_std
- mode_minmax
- mode_std
- const_minmax
- const_std
- knn_minmax
- knn_std

Przykład: 5 pierwszych wierszy dla 'original_std' (Standaryzacja):


,having_IP_Address,URL_Length,Shortining_Service,having_At_Symbol,double_slash_redirecting,Prefix_Suffix,having_Sub_Domain,SSLfinal_State,Domain_registeration_length,Favicon,...,popUpWidnow,Iframe,age_of_domain,DNSRecord,web_traffic,Page_Rank,Google_Index,Links_pointing_to_page,Statistical_report,Result
0,-1.383621,2.131846,0.387596,0.419581,-2.595298,-0.390832,-1.301443,-1.371793,-0.704342,0.477535,...,0.489496,0.317423,-1.063187,-1.48683,-1.555200,-0.589894,0.402136,1.150977,-2.476228,-1
1,0.722676,2.131846,0.387596,0.419581,0.385277,-0.390832,-0.078228,0.821449,-0.704342,0.477535,...,0.489496,0.317423,-1.063187,-1.48683,-0.347081,-0.589894,0.402136,1.150977,0.403804,-1
2,0.722676,0.826526,0.387596,0.419581,0.385277,-0.390832,-1.301443,-1.371793,-0.704342,0.477535,...,0.489496,0.317423,0.940483,-1.48683,0.861037,-0.589894,0.402136,-0.603581,-2.476228,-1
3,0.722676,0.826526,0.387596,0.419581,0.385277,-0.390832,-1.301443,-1.371793,1.419636,0.477535,...,0.489496,0.317423,-1.063187,-1.48683,0.861037,-0.589894,0.402136,-2.358139,0.403804,-1
4,0.722676,0.826526,-2.579770,0.419581,0.385277,-0.390832,1.144986,0.821449,-0.704342,0.477535,...,-2.042734,0.317423,-1.063187,-1.48683,-0.347081,-0.589894,0.402136,1.150977,0.403804,1


In [8]:
url_features = ['URL_Length', 'having_At_Symbol', 'Prefix_Suffix', 'Abnormal_URL']

df_original['URL_Risk_Score'] = df_original[url_features].sum(axis=1)

print("Wygląd nowej cechy 'URL_Risk_Score' obok cech składowych (5 pierwszych wierszy):")
display(df_original[url_features + ['URL_Risk_Score', 'Result']].head())

korelacja = df_original['URL_Risk_Score'].corr(df_original['Result'])
print(f"\nKorelacja nowej cechy (URL_Risk_Score) z wynikiem (Result) wynosi: {korelacja:.2f}")

Wygląd nowej cechy 'URL_Risk_Score' obok cech składowych (5 pierwszych wierszy):


,URL_Length,having_At_Symbol,Prefix_Suffix,Abnormal_URL,URL_Risk_Score,Result
0,1,1,-1,-1,0,-1
1,1,1,-1,1,2,-1
2,0,1,-1,-1,-1,-1
3,0,1,-1,1,1,-1
4,0,1,-1,1,1,1



Korelacja nowej cechy (URL_Risk_Score) z wynikiem (Result) wynosi: 0.19
